### STEP 1: Importing Data
1.1.Import the data set into the Python environment. Note that this is a .txt file with tab delimiters. 


In [6]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neural_network import MLPClassifier

df = pd.read_csv(r"C:\Users\annac\Downloads\RiskComplete.txt", sep="\t")

### STEP 2: Data Cleaning
2.1. Duplicates: Use the ID attribute to check for duplicate observations. If there are any, keep only a
single observation, while removing the rest.

2.2. Dropping attributes: After the 1st step, remove the “ID” attribute from the data as it is a unique
identifier and has no prediction value.

2.3. Missing values:
Check for missing values (e.g., null, nan, etc.). Notice that there may be missing values in several
attributes

In [7]:
df = df.drop_duplicates(subset=["ID"])
df = df.drop(columns=["ID"])
missing_counts = df.isnull().sum()

QUESTION #1: Which attributes have missing values and how many?

• For numeric attributes (integer or float type), replace missing values with the attribute average. Round the average values to the nearest integer before replacing the missing values.

• For string attributes (string / object type), replace missing values with the most frequent value in
the attribute column.

In [8]:
print(df.isnull().sum())

num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(round(df[num_cols].mean()))

text_cols = df.select_dtypes(include='object').columns
df[text_cols] = df[text_cols].fillna(df[text_cols].mode().iloc[0])

AGE         0
INCOME      3
GENDER      4
MARITAL     0
NUMKIDS     0
NUMCARDS    4
HOWPAID     0
MORTGAGE    0
STORECAR    0
LOANS       5
VALUE       0
dtype: int64


QUESTION #2: After replacing the missing values, what is the mean of the “LOANS” attribute? Provide
your answer in 4 decimal precision (e.g., 1.1234).

In [9]:
loans_mean = df["LOANS"].mean()
print("Mean of LOANS:", round(loans_mean, 4))

Mean of LOANS: 1.3758


2.4. Dropping observations:

• Some of the values in the data are labeled as “unknown”. Note that this happens in multiple
columns.

• Drop the entire observation (i.e., entire row) from the data, if any attribute value of the
observation is “unknown”.


In [10]:
df = df[~df.isin(["unknown"]).any(axis=1)]

QUESTION #3: After these operations, how many observations remain in the data set?

In [11]:
print("Remaining Observations:", len(df))

Remaining Observations: 4100


### STEP 3: Descriptive Statistics

QUESTION #4: What is the ratio of female (“f”) observations in the “GENDER” attribute? Provide your
answer in 4 decimal precision (e.g., 0.5123).

In [13]:
female_count = len(df[df["GENDER"] == "f"])
total_count = len(df)
female_ratio = female_count / total_count
print("Female Ratio:", round(female_ratio, 4))

Female Ratio: 0.5049


QUESTION #5: Provide the frequency distribution of the “VALUE” attribute (e.g., strong profit: 1234,
weak profit: 123, etc.)

In [14]:
print("VALUE Distribution")
print(df["VALUE"].value_counts())

VALUE Distribution
VALUE
weak profit      2396
bad loss          901
strong profit     803
Name: count, dtype: int64


### STEP 4: Data Transformation

4.1. Re-classify the target attribute: The target attribute is “VALUE”. Currently, it is multi-class with 3
possible values – 2 for profit and 1 for loss. Suppose that we’d like to represent this information using
only 2 possible values: “Profit” and “Loss”. Reclassify the values so that the label for any kind of profit is
“Profit” and any kind of loss is “Loss”. 

In [15]:
def reclassify_value(val):
    if "profit" in str(val).lower():
        return "Profit"
    else:
        return "Loss"

df["VALUE"] = df["VALUE"].apply(reclassify_value)

QUESTION #6: Provide the frequency distribution of the “VALUE” attribute after re-classification.

In [16]:
print("Re-classified VALUE Distribution")
print(df["VALUE"].value_counts())

Re-classified VALUE Distribution
VALUE
Profit    3199
Loss       901
Name: count, dtype: int64


4.2. Numerical encoding of string attributes: Currently, there are 5 attributes in the data that are stored
in string (object) format, including the target attribute.

• Encode the 4 input attributes using 1/0 dummy coding. Make sure to drop one of the newly
generated dummies from the data after the encoding operation (Hint: if using the Pandas’
get_dummies function, you need to pass a specific parameter to do this).

• Re-code the target attribute (VALUE) using 1/0 mapping. Set the value as 1 if the observation
was a “profit” and set the value as 0 if it was a “loss”.

In [17]:
df = pd.get_dummies(df, columns=["GENDER", "MARITAL", "HOWPAID", "MORTGAGE"], drop_first=True)
df["VALUE"] = df["VALUE"].apply(lambda val: 1 if val == "Profit" else 0)

4.3. Scaling of numeric attributes: There are 6 numeric attributes: AGE, INCOME, NUMKIDS,
NUMCARDS, STORECAR, and LOANS. It is a good practice to scale (standardize or normalize) these
attributes before using them as inputs in a neural network model.

• Standardize the following two attributes: [AGE and INCOME]. Standardization refers to rescaling of the values so that their mean is 0 and standard deviation is 1.
    o To do this, use the StandardScaler class from the sklearn.preprocessing package
    o Remember that data transformation functions in scikit-learn require instantiation of an object first – e.g., stScaler = StandardScaler().
    o Once this object is created, you can call “fit” and “transform” methods on it to scale the
columns that you want.

• Normalize the following four attributes: [NUMKIDS, NUMCARDS, STORECAR, and LOANS].
Normalization refers to rescaling of the values linearly so that the entire value range is between
0 and 1. Mathematically, this is 𝑥
𝑛𝑒𝑤 = (𝑥
𝑜𝑙𝑑 − min (𝑋)/(max(X) − min(X)).
    o To do this, you can use the MinMaxScaler class from sklearn.preprocessing package
    o Similar to the previous function, you need to instantiate an object and call fit and transform methods.


In [18]:
stScaler = StandardScaler()
df[["AGE", "INCOME"]] = stScaler.fit_transform(df[["AGE", "INCOME"]])

mmScaler = MinMaxScaler()
cols_to_norm = ["NUMKIDS", "NUMCARDS", "STORECAR", "LOANS"]
df[cols_to_norm] = mmScaler.fit_transform(df[cols_to_norm])

QUESTION #7: What is the mean of the standardized INCOME attribute?

In [19]:
print("Mean of standardized INCOME:", round(df['INCOME'].mean(), 4))

Mean of standardized INCOME: 0.0


QUESTION #8: What is the mean of the normalized NUMCARDS attribute?

In [20]:
print("Mean of normalized NUMCARDS:", round(df['NUMCARDS'].mean(), 4))

Mean of normalized NUMCARDS: 0.4049


STEP 5: Model Building 
5.1. Data Set Prep: Split the data into training input (X) and training label (y) data sets. If done correctly, the two data sets should look like the screenshots below. 


In [21]:
X = df.drop(columns=["VALUE"])
y = df["VALUE"]

5.2. Instantiating a Neural Network model: Instantiate a multi-layer NN model using the Multi-layer Perceptron algorithm implementation in scikit-learn. This implementation can be imported as: "from sklearn.neural_network import MLPClassifier”. Create an MLPClassifier object as the empty model and configure its parameters as follows: 

•	3 hidden layers 
•	Each hidden layer has the same number of neurons as the number of input attributes in the training data 
•	To set the hidden layer / neuron sizes, you can use the “hidden_layer_sizes” method. 
•	Activation function: logistic – i.e., sigmoid function 
•	Solver: stochastic gradient descent (sgd) 
•	Learning rate (learning_rate_init): 0.3 
•	Random_state = 1 and shuffle = False 

In [23]:
num_inputs = len(X.columns)
mlp = MLPClassifier(
    hidden_layer_sizes=(num_inputs, num_inputs, num_inputs),
    activation='logistic',
    solver='sgd',
    learning_rate_init=0.3,
    random_state=1,
    shuffle=False )

5.3. Training the model: Use the fit method to train the model created above using the training data set.  In addition, time the execution of the training process.  
For measuring the runtime of a code snippet, you can use Python’s time package. The “time” method in this package returns the current time in seconds. You can assign the current time to any variable at any stage of the program and find the difference between different variables to get the time elapsed. An example code snippet to do this is as follows: 

In [24]:
start_time = time.time()
mlp.fit(X, y)
end_time = time.time()

Training Time (seconds): 0.8972


QUESTION #9: How long did it take to train this NN model? 

In [26]:
training_time = end_time - start_time
print("Training Time (seconds):", round(training_time, 4))

Training Time (seconds): 0.8972


QUESTION #10: Answer the following questions: 
•	How many total layers are in the model? 
•	How many output nodes in the model? 
•	What is the weight of the link between the 10th input (index 9) and the 1st neuron (index 0) in the 1st hidden layer? 
Hint: You can use the attributes of the MLPClassifier listed on the following page to answer these questions: https://scikit-
learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html  

In [27]:
print("Total layers:", mlp.n_layers_)
print("Output nodes:", mlp.n_outputs_)
print("Weight (10th input to 1st hidden neuron):", mlp.coefs_[0][9][0])

Total layers: 5
Output nodes: 1
Weight (10th input to 1st hidden neuron): 0.3473037394356415


QUESTION #11: Change the configuration of the NN model to have 500 neurons in each layer. Keep everything else as is. Re-train the model using this configuration and find out how long it took to train this new NN model. Do you think that this model will perform better than the one from Step 5.2 (in terms of prediction performance)? 

In [28]:
mlp_500 = MLPClassifier(
    hidden_layer_sizes=(500, 500, 500),
    activation='logistic',
    solver='sgd',
    learning_rate_init=0.3,
    random_state=1,
    shuffle=False
)

start_time_500 = time.time()
mlp_500.fit(X, y)
end_time_500 = time.time()

training_time_500 = end_time_500 - start_time_500
print("Training Time with 500 neurons (seconds):", round(training_time_500, 4))

Training Time with 500 neurons (seconds): 5.5281
